# Bài 1 — Chunking và Embedding

**Tuần 20** · Bài tập kèm theo: `ex01_chunking.py`

Bài này trả lời hai câu hỏi:

1. **Cắt tài liệu thế nào?** — và cái giá của từng cách cắt
2. **Biến chữ thành số thế nào?** — để máy tìm được theo *ý nghĩa*, không chỉ theo từ

> Chạy được **không cần API key và không cần tải model**. Nếu bạn đã cài
> `pip install -e ".[rag]"` thì có thêm phần so sánh với model embedding thật.

Trước khi chạy, tạo dữ liệu:

```powershell
python curriculum/08-rag/tao_du_lieu.py
```

In [ ]:
import sys
from pathlib import Path

import numpy as np

# Tim thu muc goc du an du notebook duoc mo tu dau
GOC = Path.cwd()
while not (GOC / "pyproject.toml").exists() and GOC != GOC.parent:
    GOC = GOC.parent
P08 = GOC / "curriculum" / "08-rag"
sys.path.insert(0, str(P08))

from tien_ich import bat_utf8, nap_tai_lieu, nap_cau_hoi_vang
from nhung_gia_lap import nhung, nhung_nhieu, giai_thich, SO_CHIEU, TEN_CHIEU

bat_utf8()   # Windows: console cp1252 se lam sap moi lenh print tieng Viet

DOCS = nap_tai_lieu()
VANG = nap_cau_hoi_vang()

print(f"{len(DOCS)} tài liệu · {sum(len(v) for v in DOCS.values()):,} ký tự")
print(f"{len(VANG)} câu hỏi vàng · embedding giả lập {SO_CHIEU} chiều")

## 0. Nhìn vào dữ liệu trước đã

Thói quen của Phase 03 vẫn đúng ở đây: **đừng xây gì trước khi nhìn dữ liệu.**

In [ ]:
for ten, noi_dung in DOCS.items():
    dong_dau = noi_dung.split("\n")[0]
    print(f"{ten:<32} {len(noi_dung):>6,} ký tự   {dong_dau}")

In [ ]:
# Bộ câu hỏi vàng: mỗi câu kèm ĐOẠN TRÍCH chứa đáp án
from collections import Counter

print(Counter(c["loai"] for c in VANG), "\n")
for c in VANG[:3] + VANG[-2:]:
    print(f'[{c["loai"]:<14}] {c["cau_hoi"]}')
    print(f'   trích dẫn đúng: {c["trich_dan"]}')

> 📌 Hai câu cuối có `trich_dan` **rỗng** — đó là các câu **không có đáp án
> trong tài liệu**. Với chúng, câu trả lời đúng là *"không tìm thấy"*.
> Chúng ta sẽ quay lại chỗ này ở bài 3 và bài 4, vì đó là phần dễ hỏng nhất.

## 1. Vì sao không nhét cả kho vào prompt?

Kho tài liệu này bé xíu. Hãy tính thử chi phí nếu cứ gửi hết mỗi lần hỏi.

In [ ]:
TONG_KY_TU = sum(len(v) for v in DOCS.values())

# Ước lượng thô cho tiếng Việt: ~3 ký tự / token.
# Con số CHÍNH XÁC phải lấy từ client.messages.count_tokens() (Phase 07),
# ở đây chỉ cần một ước lượng để so sánh độ lớn.
TOKEN_UOC = TONG_KY_TU / 3
GIA_VAO = 1.00 / 1_000_000        # Haiku 4.5: $1.00 / 1M token vào

print(f"Kho hiện tại:        {TONG_KY_TU:>10,} ký tự  ~ {TOKEN_UOC:>8,.0f} token")
print(f"Chi phí 1 câu hỏi:   ${TOKEN_UOC * GIA_VAO:.5f}")
print(f"Chi phí 1.000 câu:   ${TOKEN_UOC * GIA_VAO * 1000:.2f}")
print()

# Bây giờ giả sử công ty có 500 trang tài liệu (~2.000 ký tự/trang)
LON = 500 * 2000 / 3
print(f"Nếu kho là 500 trang: ~{LON:,.0f} token")
print(f"   1.000 câu hỏi:     ${LON * GIA_VAO * 1000:.2f}")
print(f"   RAG (5 đoạn 600 ký tự ~ 1.000 token): ${1000 * GIA_VAO * 1000:.2f}")

Chênh lệch hai dòng cuối chính là lý do RAG tồn tại.

Nhưng hãy công bằng: với kho **hiện tại** (9.500 ký tự), nhét hết vào prompt
kèm **prompt caching** còn rẻ và chính xác hơn RAG — vì không có bước tìm kiếm
nào để mà sai.

> ⚠️ Rất nhiều dự án dựng cả hệ RAG cho một tập tài liệu mà lẽ ra chỉ cần
> `cache_control`. Trước khi xây, hãy hỏi: **kho của tôi có thật sự lớn không?**

## 2. Cắt theo kích thước cố định

Cách đơn giản nhất. Và cũng là cách để lộ vấn đề rõ nhất.

In [ ]:
def cat_co_dinh(van_ban, kich_thuoc, chong_lap=0):
    """Giong ex01_chunking.cat_co_dinh - dat o day de notebook chay doc lap."""
    if chong_lap >= kich_thuoc:
        raise ValueError("chong_lap phai nho hon kich_thuoc")
    buoc = kich_thuoc - chong_lap
    return [van_ban[i:i + kich_thuoc] for i in range(0, len(van_ban), buoc)]


def cat_kho(docs, kich_thuoc=600, chong_lap=120):
    """Cat toan bo kho -> (danh sach chunk, danh sach ten file tuong ung)."""
    chunks, nguon = [], []
    for ten, noi_dung in docs.items():
        for c in cat_co_dinh(noi_dung, kich_thuoc, chong_lap):
            chunks.append(c)
            nguon.append(ten)
    return chunks, nguon


def gop(s):
    """Gop moi khoang trang/xuong dong thanh mot dau cach."""
    return " ".join(s.split())


def co_trich_dan(cac_chunk, trich_dan):
    """Doan trich dan co nam trong cac chunk lay ve khong?"""
    kho = gop(" || ".join(cac_chunk))
    return gop(trich_dan) in kho

In [ ]:
vb = DOCS["chinh_sach_nghi_phep.md"]
cac_chunk = cat_co_dinh(vb, 300, 0)

print(f"{len(vb)} ký tự -> {len(cac_chunk)} chunk\n")
for i, c in enumerate(cac_chunk[:3]):
    print(f"--- chunk {i} ({len(c)} ký tự) ---")
    print(gop(c)[:120] + "...")
    print()

### Chỗ nối giữa hai chunk

Nhìn kỹ **20 ký tự cuối chunk 0 và 20 ký tự đầu chunk 1**:

In [ ]:
for i in range(3):
    print(f"cuối chunk {i}   ...{gop(cac_chunk[i][-40:])!r}")
    print(f"đầu  chunk {i+1}  {gop(cac_chunk[i+1][:40])!r}...")
    print()

Câu bị cắt làm đôi. Nếu đáp án nằm đúng chỗ nối này, **không chunk nào chứa
đủ nó** — và retrieval sẽ không bao giờ lấy được câu trả lời hoàn chỉnh.

Đây không phải chuyện lý thuyết. Hãy đo trên bộ câu hỏi vàng.

In [ ]:
def dem_trich_dan_bi_cat(docs, kich_thuoc, chong_lap):
    """Bao nhiêu đoạn trích đáp án KHÔNG nằm trọn trong bất kỳ chunk nào?"""
    chunks, _ = cat_kho(docs, kich_thuoc, chong_lap)
    bi_cat = []
    for c in VANG:
        for td in c["trich_dan"]:
            if not any(gop(td) in gop(ch) for ch in chunks):
                bi_cat.append((c["id"], td))
    return len(chunks), bi_cat


for kt, cl in [(300, 0), (300, 60), (600, 0), (600, 120)]:
    n, bi_cat = dem_trich_dan_bi_cat(DOCS, kt, cl)
    print(f"chunk {kt:>4} / chồng lấn {cl:>3}  ->  {n:>2} chunk, "
          f"{len(bi_cat)} đoạn trích bị cắt đôi")
    for ma_id, td in bi_cat:
        print(f"      ✂  [{ma_id}] {td!r}")

> 📌 **Chồng lấn không phải trang trí.** Nó là thứ đảm bảo mọi câu nằm trọn
> vẹn trong ít nhất một chunk.
>
> Giá phải trả: chồng lấn 20% làm kho vector to hơn ~25%. Bạn đang **đổi bộ nhớ
> và tiền lấy recall** — một đánh đổi hoàn toàn hợp lý, miễn là bạn biết mình
> đang đổi.

## 3. Ba cách cắt, ba dáng hình khác nhau

In [ ]:
def cat_theo_doan(van_ban, toi_da):
    cac_doan = [d.strip() for d in van_ban.split("\n\n") if d.strip()]
    kq, dem, do_dai = [], [], 0
    for doan in cac_doan:
        them = len(doan) + (2 if dem else 0)
        if dem and do_dai + them > toi_da:
            kq.append("\n\n".join(dem))
            dem, do_dai, them = [], 0, len(doan)
        dem.append(doan)
        do_dai += them
    if dem:
        kq.append("\n\n".join(dem))
    return kq


def cat_theo_tieu_de(van_ban, muc=2):
    dau = "#" * muc + " "
    kq, hien_tai = [], []
    for dong in van_ban.split("\n"):
        if dong.startswith(dau) and dong[muc:muc + 1] != "#" and hien_tai:
            kq.append("\n".join(hien_tai).strip())
            hien_tai = []
        hien_tai.append(dong)
    if hien_tai:
        kq.append("\n".join(hien_tai).strip())
    return [c for c in kq if c]


def thong_ke(cac_chunk):
    d = [len(c) for c in cac_chunk]
    return {"so": len(d), "tb": sum(d) / len(d), "min": min(d), "max": max(d)}


print(f'{"cách cắt":<26}{"số":>5}{"t.bình":>9}{"ngắn":>7}{"dài":>7}')
for ten, f in [("cố định 600/120", lambda: cat_co_dinh(vb, 600, 120)),
               ("theo đoạn (600)", lambda: cat_theo_doan(vb, 600)),
               ("theo tiêu đề ##", lambda: cat_theo_tieu_de(vb))]:
    t = thong_ke(f())
    print(f'{ten:<26}{t["so"]:>5}{t["tb"]:>9.0f}{t["min"]:>7}{t["max"]:>7}')

Cột `ngắn` và `dài` là chỗ đáng nhìn nhất:

- **Cố định** cho chunk đều nhau → chi phí mỗi câu hỏi dự đoán được
- **Theo đoạn / tiêu đề** cho chunk lệch nhau rất nhiều → một chunk khổng lồ có
  thể một mình chiếm hết ngân sách token của prompt

> Chunk quá ngắn (vài chục ký tự) gần như chắc chắn là rác — một dòng tiêu đề
> lạc, một dòng kẻ ngang. Chúng vẫn chiếm chỗ trong top-k và **đẩy đoạn có ích
> ra ngoài**. Nhìn cột `ngắn` là biết ngay.

## 4. Cái bẫy của việc cắt theo tiêu đề

Cắt theo tiêu đề nghe rất hợp lý: mỗi chunk là một đơn vị ý nghĩa trọn vẹn.

Hãy thử với tài liệu kỹ thuật, câu hỏi *"làm sao sửa lỗi VX-204?"*.

In [ ]:
kt = DOCS["huong_dan_ky_thuat.md"]
theo_de = cat_theo_tieu_de(kt)

for i, c in enumerate(theo_de):
    tieu_de = c.split("\n")[0]
    co_ma = "VX-204" in c
    co_cach_sua = "đăng xuất khỏi ứng dụng" in c
    print(f'[{i}] {tieu_de:<42} VX-204:{"CÓ" if co_ma else "  -"}'
          f'   cách sửa:{"CÓ" if co_cach_sua else "  -"}')

**Nhìn hai cột cuối: chúng không bao giờ cùng "CÓ" trên một dòng.**

Tài liệu viết mô tả lỗi ở mục 2, còn mục 3 mở đầu bằng *"Với lỗi ở mục trên..."*
— **không nhắc lại chuỗi `VX-204`**. Cắt theo tiêu đề vừa cắt đứt nguyên nhân
khỏi cách sửa.

Hệ quả: tìm "VX-204" sẽ ra chunk mô tả lỗi, và câu trả lời sẽ là *"đây là lỗi
xung đột dữ liệu chấm công"* — **đúng nhưng vô dụng**, vì người hỏi muốn biết
cách sửa.

> ⚠️ Đây là kiểu hỏng **im lặng** đặc trưng của RAG: không có exception nào,
> câu trả lời vẫn trôi chảy, chỉ là nó thiếu mất nửa quan trọng. Không có bộ
> câu hỏi vàng thì bạn sẽ không bao giờ phát hiện.

Câu `sua_loi_vx204` trong bộ test tồn tại chính vì lý do này — nó đòi **hai**
đoạn trích cùng lúc.

## 5. Embedding — biến chữ thành số

Ta cần một cách để *"ngồi nhà làm việc"* và *"làm việc từ xa"* ở gần nhau
trong không gian số, dù không chung một từ nào.

> **Anthropic không có endpoint embedding** — Claude sinh văn bản, không trả
> vector. Bạn dùng `sentence-transformers` chạy máy mình (miễn phí), Voyage AI,
> hoặc nhà cung cấp khác. Notebook này dùng **bản giả lập offline** để chạy
> được ngay.

In [ ]:
cau_a = "Tôi có được ngồi nhà làm việc không?"
cau_b = "Chính sách làm việc từ xa thế nào?"
cau_c = "Sản phẩm bảo hành bao lâu?"

for cau in (cau_a, cau_b, cau_c):
    print(f"{cau}\n   {giai_thich(cau)}")

print()
print(f"cosine(a, b) = {nhung(cau_a) @ nhung(cau_b):.3f}   <- cùng ý, khác từ")
print(f"cosine(a, c) = {nhung(cau_a) @ nhung(cau_c):.3f}   <- khác ý")

Không một từ nào chung giữa hai câu đầu, nhưng vector của chúng trùng nhau.
Đó là **toàn bộ** giá trị của embedding so với tìm theo từ khoá.

### Bản giả lập này hoạt động ra sao — và mù ở đâu

Mỗi chiều là một **khái niệm** (`tu_xa`, `bao_hanh`, `phep_nam`...); giá trị là
số lần văn bản nhắc tới khái niệm đó. Model thật **học** quan hệ này từ hàng tỷ
câu; ở đây ta **gán tay** bằng một từ điển 27 khái niệm.

In [ ]:
# Chỗ bản giả lập mù tịt: từ nằm ngoài từ điển khái niệm
for cau in ["Mã lỗi VX-204 nghĩa là gì?",
            "Công ty có nuôi mèo văn phòng không?"]:
    v = nhung(cau)
    print(f'{cau}\n   khái niệm: {giai_thich(cau)}  |  vector toàn 0: {not v.any()}')

Câu hỏi về **mã lỗi** gần như không có tín hiệu.

Điều đáng nói: đây **không chỉ là giới hạn của bản giả lập**. Embedding thật
cũng yếu với chuỗi định danh hiếm — model chưa từng thấy `VX-204` đủ nhiều lần
để học ý nghĩa của nó. Bài 2 sẽ đo chính xác chỗ này và cho bạn cách khắc phục.

In [ ]:
# Nếu bạn đã cài  pip install -e ".[rag]"  thì so sánh với model thật
try:
    from sentence_transformers import SentenceTransformer

    print("Đang tải model (lần đầu ~90 MB)...")
    mt = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    va, vb, vc = mt.encode([cau_a, cau_b, cau_c], normalize_embeddings=True)
    print(f"Model thật ({va.shape[0]} chiều):")
    print(f"   cosine(a, b) = {va @ vb:.3f}   <- cùng ý")
    print(f"   cosine(a, c) = {va @ vc:.3f}   <- khác ý")
    print("   (Giá trị tuyệt đối THẤP hơn hẳn bản giả lập - đừng so hai bên "
          "với nhau.")
    print("    Điều duy nhất cần đúng là: cùng ý > khác ý.)")
except Exception as e:
    print(f"⏭  Bỏ qua model thật ({type(e).__name__}). Dùng bản giả lập là đủ "
          f"cho mọi bài học ở đây.")
    print('   Muốn thử: pip install -e ".[rag]"')

## 6. Đo: chunk size ảnh hưởng recall thế nào?

Đây là phần quan trọng nhất của bài. Ta đã có bộ câu hỏi vàng — hãy **đo**
thay vì đoán.

In [ ]:
def do_recall(kich_thuoc, chong_lap, k=3):
    """Recall trung bình khi tìm bằng embedding, đo ở mức ĐOẠN TRÍCH."""
    chunks, _ = cat_kho(DOCS, kich_thuoc, chong_lap)
    M = nhung_nhieu(chunks)

    diem_tung_cau = []
    for c in VANG:
        if not c["trich_dan"]:          # câu không có đáp án -> không đo recall
            continue
        top = np.argsort(-(M @ nhung(c["cau_hoi"])))[:k]
        lay = [chunks[i] for i in top]
        dat = sum(1 for td in c["trich_dan"] if co_trich_dan(lay, td))
        diem_tung_cau.append(dat / len(c["trich_dan"]))

    return len(chunks), sum(diem_tung_cau) / len(diem_tung_cau)


print(f'{"cấu hình":<24}{"số chunk":>10}{"recall@3":>10}{"token/câu hỏi":>16}')
for kt, cl in [(300, 0), (300, 60), (600, 120), (1000, 200)]:
    n, r = do_recall(kt, cl)
    print(f'{f"{kt} ký tự / chồng {cl}":<24}{n:>10}{r:>10.2f}{kt * 3 / 3:>16,.0f}')

### Một kết quả trông có vẻ mâu thuẫn

Nhìn hai dòng đầu: `300/60` **không còn đoạn trích nào bị cắt đôi** (mục 2 vừa
chứng minh), vậy mà recall lại **thấp hơn** `300/0`.

Không mâu thuẫn — chồng lấn tạo ra các chunk **gần trùng nhau**. Ba chunk chồng
lấn cùng nói về nghỉ ốm sẽ cùng ghi điểm cao và **chiếm cả ba suất trong top-3**,
đẩy đoạn về một chủ đề khác (mà câu hỏi cũng cần) ra ngoài.

> 📌 Bài học không phải "đừng chồng lấn", mà là: **hai chỉ số tốt lên không đồng
> nghĩa với nhau.** Chồng lấn cải thiện *tính toàn vẹn của chunk* nhưng làm hỏng
> *tính đa dạng của top-k*. Bạn phải đo cái mình thật sự quan tâm.
>
> Cách xử lý phổ biến: lấy rộng hơn (k=10) rồi **loại chunk gần trùng** trước
> khi đưa vào prompt. Đó là bài tập nâng cao ở mini-project.

Hai điều đọc được:

1. **Chunk to hơn → recall cao hơn.** Đoạn chứa đáp án ít bị cắt rời hơn.
2. Nhưng cột cuối cho thấy cái giá: chunk 1000 ký tự × 3 đoạn ≈ **1.000 token
   mỗi câu hỏi**, gấp ba lần chunk 300.

> 📌 **Không có chunk size "đúng".** Chỉ có chunk size bạn đã **đo** trên tài
> liệu và câu hỏi của chính mình. Ai bảo bạn "cứ để 512 token là chuẩn" thì
> người đó chưa đo bao giờ.
>
> Và recall chưa phải toàn bộ câu chuyện — bài 3 sẽ cho thấy nhồi **nhiều đoạn
> hơn** có thể làm câu trả lời **tệ đi**.

## Tóm tắt

| Điều | Nhớ gì |
|---|---|
| **Khi nào cần RAG** | Kho lớn/hay đổi. Kho nhỏ và tĩnh → prompt caching rẻ và chính xác hơn |
| **Chồng lấn** | Không phải trang trí — nó cứu các câu nằm vắt qua ranh giới |
| **Cắt theo tiêu đề** | Gọn gàng, nhưng cắt đứt "xem mục trên" |
| **Chunk ngắn bất thường** | Gần như chắc chắn là rác, và nó chiếm chỗ trong top-k |
| **Embedding** | Bắt được ý; mù với mã định danh hiếm |
| **Chunk size** | Phải **đo**, không phải chọn theo lời khuyên trên mạng |

**Bài tập:** `curriculum/08-rag/exercises/ex01_chunking.py`

```powershell
pytest tests/phase08/test_ex01.py -v
```

➡️ Bài tiếp: **[02 — Tìm kiếm: BM25, vector và hybrid](02_tim_kiem_va_hybrid.ipynb)**